# RappiPlus — Análisis de negocio con Python, SQL y Power BI

Proyecto de formación de José Luis Monsálvez, Bootcamp Data Analytics de TripleTen.

Esta copia conserva los cálculos y resultados guardados del notebook original. La configuración de conexión se realiza mediante `RAPPIPLUS_DATABASE_URL`. Consulta el README para los datos, dependencias y límites de ejecución.

**Resultados guardados:** 24.920 pedidos limpios; resultado después de costes de productos y marketing de 2.910.708,05 (30,27 % de ingresos); retención semanal cercana al 42 %; test A/B sin diferencia significativa (p = 0,4161).

La rentabilidad considera solo los costes disponibles. El funnel usa conteos independientes por evento, sin validar un recorrido secuencial individual.


# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [6]:
# cargar archivos
# cargar archivos
orders = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv'
)

catalog = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv'
)

marketing = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv'
)
  


In [7]:
# explorar datasets
print('Dimensiones de orders:', orders.shape)
display(orders.head())

Dimensiones de orders: (25100, 12)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [8]:
print('Dimensiones de catalog:', catalog.shape)
display(catalog.head())

Dimensiones de catalog: (7, 4)


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [9]:
print('Dimensiones de marketing:', marketing.shape)
display(marketing.head())

Dimensiones de marketing: (1620, 5)


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


### Conclusión de la carga inicial

Los tres datasets se cargaron correctamente. El dataset `orders` contiene 25.100 filas y 12 columnas; `catalog`, 7 filas y 4 columnas; y `marketing`, 1.620 filas y 5 columnas.

La revisión inicial confirma que las columnas y la estructura general coinciden con la documentación del proyecto. También se observa una posible inconsistencia en la escritura de la categoría `Electronica`/`Electrónica`, que será revisada durante el proceso de limpieza.

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [10]:
print('INFORMACIÓN DE ORDERS')
orders.info()

INFORMACIÓN DE ORDERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [11]:
print('\nINFORMACIÓN DE CATALOG')
catalog.info()


INFORMACIÓN DE CATALOG
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [12]:
print('\nINFORMACIÓN DE MARKETING')
marketing.info()


INFORMACIÓN DE MARKETING
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


### Análisis inicial de tipos de datos y valores ausentes

El dataset `orders` presenta valores ausentes en las columnas `pais`, `dispositivo`, `fuente_referencia`, `nombre_producto`, `categoria_producto`, `cantidad`, `precio_unitario` y `monto_descuento`.

El dataset `catalog` no presenta valores ausentes y sus tipos de datos son inicialmente coherentes.

El dataset `marketing` presenta 101 valores ausentes en la columna `canal`.

Las columnas `fecha_hora_pedido` de `orders` y `fecha` de `marketing` fueron cargadas como tipo `object`, por lo que deberán convertirse al formato de fecha antes del análisis temporal.

La columna `cantidad` se encuentra como `float64`. Antes de convertirla a un tipo entero será necesario investigar y tratar sus valores ausentes e inválidos.

In [13]:
# contabilizar valores ausentes
print('VALORES AUSENTES EN ORDERS')
display(orders.isna().sum().to_frame(name='cantidad_nulos'))

VALORES AUSENTES EN ORDERS


,cantidad_nulos
id_pedido,0
id_usuario,0
fecha_hora_pedido,0
pais,300
dispositivo,20
fuente_referencia,30
nombre_producto,30
categoria_producto,80
cantidad,50
precio_unitario,50


In [14]:
print('VALORES AUSENTES EN CATALOG')
display(catalog.isna().sum().to_frame(name='cantidad_nulos'))

VALORES AUSENTES EN CATALOG


,cantidad_nulos
nombre_producto,0
categoria_producto,0
costo_unitario,0
proveedor,0


In [15]:
print('VALORES AUSENTES EN MARKETING')
display(marketing.isna().sum().to_frame(name='cantidad_nulos'))

VALORES AUSENTES EN MARKETING


,cantidad_nulos
fecha,0
pais,0
id_campaña,0
canal,101
gasto,0


In [16]:
# convertir columnas de fecha
orders['fecha_hora_pedido'] = pd.to_datetime(
    orders['fecha_hora_pedido'],
    errors='coerce'
)

marketing['fecha'] = pd.to_datetime(
    marketing['fecha'],
    errors='coerce'
)

# comprobar el resultado
print('Tipo de fecha_hora_pedido:', orders['fecha_hora_pedido'].dtype)
print('Fechas inválidas en orders:', orders['fecha_hora_pedido'].isna().sum())

print('\nTipo de fecha:', marketing['fecha'].dtype)
print('Fechas inválidas en marketing:', marketing['fecha'].isna().sum())

print('\nPeriodo de pedidos:')
print(orders['fecha_hora_pedido'].min(), '-', orders['fecha_hora_pedido'].max())

print('\nPeriodo de marketing:')
print(marketing['fecha'].min(), '-', marketing['fecha'].max())

Tipo de fecha_hora_pedido: datetime64[ns]
Fechas inválidas en orders: 0

Tipo de fecha: datetime64[ns]
Fechas inválidas en marketing: 0

Periodo de pedidos:
2025-01-01 00:00:00 - 2025-06-30 00:00:00

Periodo de marketing:
2025-01-01 00:00:00 - 2025-06-29 00:00:00


### Validación y conversión de fechas

Las columnas `fecha_hora_pedido` y `fecha` se convirtieron correctamente de `object` a `datetime64[ns]`.

No se detectaron fechas inválidas durante la conversión. El dataset de pedidos cubre el periodo comprendido entre el 1 de enero y el 30 de junio de 2025, mientras que el dataset de marketing abarca desde el 1 de enero hasta el 29 de junio de 2025.

In [17]:
# revisar estadísticas de las variables numéricas
columnas_numericas_orders = [
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]

print('ESTADÍSTICAS NUMÉRICAS DE ORDERS')
display(orders[columnas_numericas_orders].describe().T)

print('ESTADÍSTICAS NUMÉRICAS DE CATALOG')
display(catalog[['costo_unitario']].describe().T)

print('ESTADÍSTICAS NUMÉRICAS DE MARKETING')
display(marketing[['gasto']].describe().T)

ESTADÍSTICAS NUMÉRICAS DE ORDERS


,count,mean,std,min,25%,50%,75%,max
cantidad,25050.0,7.092735,296.277003,-2.00,1.0000,2.000,2.0000,20000.00
precio_unitario,25050.0,259.305549,138.726461,20.03,138.3775,258.715,380.3325,499.96
monto_descuento,25050.0,4.500798,5.223010,0.00,0.0000,0.000,10.0000,15.00
monto_total,25100.0,2072.679584,98949.949719,-492.65,180.5075,341.750,518.5800,8840200.00


ESTADÍSTICAS NUMÉRICAS DE CATALOG


,count,mean,std,min,25%,50%,75%,max
costo_unitario,7.0,102.252857,111.011563,10.12,16.905,25.21,182.975,280.68


ESTADÍSTICAS NUMÉRICAS DE MARKETING


,count,mean,std,min,25%,50%,75%,max
gasto,1620.0,1772.74292,734.43294,501.11,1128.03,1782.425,2420.685,2999.36


### Revisión inicial de variables numéricas

Las variables numéricas de `catalog` y `marketing` no presentan valores negativos ni iguales a cero que resulten evidentemente inválidos.

En `orders` se detectaron cantidades negativas y valores extremos de hasta 20.000 unidades, aunque el 75 % de los pedidos contiene como máximo 2 unidades. Estos valores distorsionan la media y la desviación estándar.

También se detectaron montos totales negativos y un valor máximo de 8.840.200. Es necesario revisar las filas correspondientes antes de decidir su tratamiento.

In [18]:
# revisar frecuencia de valores en cantidad
print('FRECUENCIA DE LOS VALORES DE CANTIDAD')
display(
    orders['cantidad']
    .value_counts(dropna=False)
    .sort_index()
    .to_frame(name='frecuencia')
)

# localizar cantidades sospechosas
cantidades_sospechosas = orders[
    (orders['cantidad'] <= 0) |
    (orders['cantidad'] > 2)
]

print('Cantidad de filas sospechosas:', cantidades_sospechosas.shape[0])

display(
    cantidades_sospechosas[
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ].sort_values('cantidad')
)

FRECUENCIA DE LOS VALORES DE CANTIDAD


,frecuencia
-2.0,1
-1.0,3
1.0,12394
2.0,12642
10000.0,6
20000.0,4
NaN,50


Cantidad de filas sospechosas: 14


,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,Phone-Pro-128GB,-2.0,101.31,10.0,-192.62
267,order_267,Phone-Pro-128GB,-1.0,43.50,5.0,-38.50
268,order_268,Phone-Pro-128GB,-1.0,497.65,5.0,-492.65
269,order_269,Phone-Pro-128GB,-1.0,423.53,0.0,-423.53
3521,order_3521,Laptop-Gaming-16GB,10000.0,43.14,0.0,431400.00
3522,order_3522,Laptop-Gaming-16GB,10000.0,280.55,0.0,2805500.00
3586,order_3586,Laptop-Gaming-16GB,10000.0,490.35,0.0,4903500.00
3643,order_3643,Laptop-Gaming-16GB,10000.0,238.15,0.0,2381500.00
3689,order_3689,Laptop-Gaming-16GB,10000.0,87.69,0.0,876900.00
3748,order_3748,Laptop-Gaming-16GB,10000.0,336.93,0.0,3369300.00


### Identificación de cantidades inválidas

La variable `cantidad` contiene principalmente valores de 1 y 2 unidades. Sin embargo, se identificaron 14 registros sospechosos: cuatro pedidos con cantidades negativas y diez pedidos con cantidades de 10.000 o 20.000 unidades.

Las cantidades extremas se encuentran únicamente en pedidos del producto `Laptop-Gaming-16GB` y parecen corresponder a valores de 1 o 2 multiplicados accidentalmente por 10.000.

También existen 50 valores ausentes en `cantidad`. Antes de aplicar correcciones o eliminar registros, se verificará la consistencia entre cantidad, precio unitario, descuento y monto total.

In [20]:
# verificar consistencia de los montos
columnas_calculo = [
    'cantidad',
    'precio_unitario',
    'monto_descuento'
]

filas_completas = orders[columnas_calculo].notna().all(axis=1)

revision_montos = orders.loc[
    filas_completas,
    [
        'id_pedido',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
].copy()

revision_montos['monto_total_calculado'] = (
    revision_montos['cantidad'] *
    revision_montos['precio_unitario'] -
    revision_montos['monto_descuento']
).round(2)

revision_montos['diferencia'] = (
    revision_montos['monto_total'] -
    revision_montos['monto_total_calculado']
).abs().round(2)

inconsistencias_monto = revision_montos[
    revision_montos['diferencia'] > 0.01
]

print(
    'Cantidad de montos inconsistentes:',
    inconsistencias_monto.shape[0]
)

display(inconsistencias_monto)

Cantidad de montos inconsistentes: 3


,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total,monto_total_calculado,diferencia
266,order_266,-2.0,101.31,10.0,-192.62,-212.62,20.0
267,order_267,-1.0,43.50,5.0,-38.50,-48.50,10.0
268,order_268,-1.0,497.65,5.0,-492.65,-502.65,10.0


### Verificación de la consistencia de los montos

Después de redondear los cálculos monetarios a dos decimales, se identificaron tres pedidos cuyo monto total no coincide con la fórmula `cantidad × precio unitario − descuento`.

Las tres inconsistencias corresponden a pedidos con cantidades negativas. También existe un cuarto pedido con cantidad negativa cuyo monto coincide matemáticamente porque no tiene descuento, aunque continúa siendo inválido desde el punto de vista del negocio.

Los pedidos con cantidades de 10.000 y 20.000 mantienen coherencia matemática entre cantidad, precio y monto total, pero sus cantidades siguen siendo anómalas y distorsionan los ingresos.


In [21]:
# investigar filas con datos numéricos ausentes
columnas_numericas_pedido = [
    'cantidad',
    'precio_unitario',
    'monto_descuento'
]

filtro_numericos_ausentes = (
    orders[columnas_numericas_pedido]
    .isna()
    .any(axis=1)
)

pedidos_numericos_incompletos = orders.loc[
    filtro_numericos_ausentes,
    [
        'id_pedido',
        'nombre_producto',
        'categoria_producto',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
]

print(
    'Cantidad de pedidos con datos numéricos ausentes:',
    pedidos_numericos_incompletos.shape[0]
)

display(pedidos_numericos_incompletos.head(10))

print('PATRONES DE VALORES AUSENTES')

display(
    orders.loc[
        filtro_numericos_ausentes,
        columnas_numericas_pedido
    ]
    .isna()
    .value_counts()
    .reset_index(name='cantidad_filas')
)

Cantidad de pedidos con datos numéricos ausentes: 50


,id_pedido,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39
79,order_79,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15
80,order_80,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18
81,order_81,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08
82,order_82,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55
83,order_83,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35


PATRONES DE VALORES AUSENTES


,cantidad,precio_unitario,monto_descuento,cantidad_filas
0,True,True,True,50


### Evaluación de los datos numéricos ausentes

Se identificaron 50 pedidos en los que `cantidad`, `precio_unitario` y `monto_descuento` están ausentes simultáneamente.

Aunque estos registros conservan el `monto_total`, no es posible reconstruir de manera confiable los tres valores faltantes, ya que distintas combinaciones de cantidad, precio y descuento podrían producir el mismo total.

Debido a que estos registros representan aproximadamente el 0,20 % del dataset y no permiten calcular correctamente los costos y la rentabilidad, se eliminarán durante la etapa de limpieza.

In [22]:
# revisar duplicados
print('DUPLICADOS EXACTOS')
print('Orders:', orders.duplicated().sum())
print('Catalog:', catalog.duplicated().sum())
print('Marketing:', marketing.duplicated().sum())

print('\nDUPLICADOS EN CLAVES PRINCIPALES')
print(
    'ID de pedido duplicados:',
    orders['id_pedido'].duplicated().sum()
)

print(
    'Productos duplicados en catalog:',
    catalog['nombre_producto'].duplicated().sum()
)

print(
    'Combinaciones fecha-campaña duplicadas en marketing:',
    marketing.duplicated(
        subset=['fecha', 'id_campaña']
    ).sum()
)

duplicados_orders = orders[
    orders.duplicated(keep=False)
].sort_values('id_pedido')

print(
    '\nFilas involucradas en duplicados de orders:',
    duplicados_orders.shape[0]
)

print(
    'Pedidos diferentes duplicados:',
    duplicados_orders['id_pedido'].nunique()
)

display(duplicados_orders.head(10))

DUPLICADOS EXACTOS
Orders: 100
Catalog: 0
Marketing: 0

DUPLICADOS EN CLAVES PRINCIPALES
ID de pedido duplicados: 100
Productos duplicados en catalog: 0
Combinaciones fecha-campaña duplicadas en marketing: 0

Filas involucradas en duplicados de orders: 200
Pedidos diferentes duplicados: 100


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10856,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
25044,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
11449,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55
25083,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55


### Revisión de registros duplicados

Se identificaron 100 filas duplicadas exactas en `orders`, correspondientes a 100 pedidos diferentes que aparecen dos veces. En total, 200 filas están involucradas: 100 registros originales y sus 100 copias.

Debido a que todos los campos de cada pareja son idénticos, las copias adicionales se eliminarán conservando una sola aparición de cada pedido.

No se encontraron duplicados en `catalog` ni en `marketing`. Tampoco se detectaron repeticiones en la clave compuesta `fecha` e `id_campaña` del dataset de marketing.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [24]:
# revisar variables categóricas
columnas_categoricas_orders = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

print('VARIABLES CATEGÓRICAS DE ORDERS')

for columna in columnas_categoricas_orders:
    print(f'\nColumna: {columna}')
    display(
        orders[columna]
        .value_counts(dropna=False)
        .to_frame(name='frecuencia')
    )

columnas_categoricas_catalog = [
    'nombre_producto',
    'categoria_producto',
    'proveedor'
]

print('\nVARIABLES CATEGÓRICAS DE CATALOG')

for columna in columnas_categoricas_catalog:
    print(f'\nColumna: {columna}')
    display(
        catalog[columna]
        .value_counts(dropna=False)
        .to_frame(name='frecuencia')
    )

columnas_categoricas_marketing = [
    'pais',
    'id_campaña',
    'canal'
]

print('\nVARIABLES CATEGÓRICAS DE MARKETING')

for columna in columnas_categoricas_marketing:
    print(f'\nColumna: {columna}')
    display(
        marketing[columna]
        .value_counts(dropna=False)
        .to_frame(name='frecuencia')
    )

VARIABLES CATEGÓRICAS DE ORDERS

Columna: pais


,frecuencia
Colombia,7520
Mexico,7502
Argentina,7291
mexico,865
colombia,823
argentina,799
NaN,300



Columna: dispositivo


,frecuencia
desktop,12759
mobile,12321
NaN,20



Columna: fuente_referencia


...

,frecuencia
Colombia,540
Argentina,540
Mexico,540



Columna: id_campaña


,frecuencia
social_Colombia,180
organic_Colombia,180
social_Mexico,180
paid_search_Mexico,180
organic_Mexico,180
paid_search_Argentina,180
social_Argentina,180
paid_search_Colombia,180
organic_Argentina,180



Columna: canal


,frecuencia
paid_search,507
organic,506
social,506
NaN,101


### Revisión de variables categóricas

En `orders` se encontraron nombres de países escritos con diferentes combinaciones de mayúsculas y minúsculas. Las variantes `argentina`, `colombia` y `mexico` deberán estandarizarse como `Argentina`, `Colombia` y `Mexico`.

Las variables `dispositivo` y `fuente_referencia` contienen categorías válidas, aunque presentan valores ausentes. Los nombres de producto coinciden con los productos disponibles en `catalog`, pero existen 30 pedidos sin producto.

La categoría `Electronica` de `orders` no coincide exactamente con `Electrónica` en `catalog`. Se utilizará la escritura del catálogo como referencia oficial.

En `marketing` se detectaron 101 valores ausentes en `canal`. Estos valores pueden recuperarse de forma confiable porque el canal está incorporado en `id_campaña`.

In [25]:
# investigar patrones de valores categóricos ausentes
columnas_categoricas_orders = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

filtro_categoricos_ausentes = (
    orders[columnas_categoricas_orders]
    .isna()
    .any(axis=1)
)

print(
    'Pedidos con al menos un dato categórico ausente:',
    filtro_categoricos_ausentes.sum()
)

print('\nPATRONES DE AUSENCIA EN ORDERS')

display(
    orders.loc[
        filtro_categoricos_ausentes,
        columnas_categoricas_orders
    ]
    .isna()
    .value_counts()
    .reset_index(name='cantidad_filas')
)

print('\nRELACIÓN ENTRE PRODUCTO Y CATEGORÍA AUSENTE')

print(
    'Categoría ausente con producto disponible:',
    (
        orders['categoria_producto'].isna() &
        orders['nombre_producto'].notna()
    ).sum()
)

print(
    'Producto y categoría ausentes simultáneamente:',
    (
        orders['nombre_producto'].isna() &
        orders['categoria_producto'].isna()
    ).sum()
)

print('\nRECUPERACIÓN DEL CANAL DE MARKETING')

marketing_revision = marketing[
    marketing['canal'].isna()
][['id_campaña', 'canal']].copy()

marketing_revision['canal_inferido'] = (
    marketing_revision['id_campaña']
    .str.rsplit('_', n=1)
    .str[0]
)

display(marketing_revision.head(10))

print(
    'Canales que pueden recuperarse:',
    marketing_revision['canal_inferido'].notna().sum()
)

Pedidos con al menos un dato categórico ausente: 400

PATRONES DE AUSENCIA EN ORDERS


,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad_filas
0,True,False,False,False,False,300
1,False,False,False,False,True,50
2,False,False,True,True,True,30
3,False,True,False,False,False,20



RELACIÓN ENTRE PRODUCTO Y CATEGORÍA AUSENTE
Categoría ausente con producto disponible: 50
Producto y categoría ausentes simultáneamente: 30

RECUPERACIÓN DEL CANAL DE MARKETING


,id_campaña,canal,canal_inferido
98,social_Argentina,NaN,social
99,organic_Mexico,NaN,organic
100,paid_search_Mexico,NaN,paid_search
101,social_Mexico,NaN,social
102,organic_Colombia,NaN,organic
103,paid_search_Colombia,NaN,paid_search
104,social_Colombia,NaN,social
105,organic_Argentina,NaN,organic
106,paid_search_Argentina,NaN,paid_search
107,social_Argentina,NaN,social


Canales que pueden recuperarse: 101


### Estrategia de limpieza

Se conservarán los pedidos que únicamente carecen de país o dispositivo, asignándoles la categoría `Desconocido`, porque estas variables no impiden calcular ingresos, costos y rentabilidad.

Se eliminarán los 50 pedidos sin cantidad, precio unitario ni descuento, debido a que no es posible reconstruir estos valores de manera confiable. También se eliminarán los 30 pedidos sin producto, ya que no pueden relacionarse con el catálogo para calcular su costo.

Las cantidades negativas se corregirán utilizando su valor absoluto. Las cantidades de 10.000 y 20.000 se dividirán entre 10.000, debido a que representan un patrón evidente de error de escala. Después se recalculará el monto total de estos pedidos.

Los países se estandarizarán y las categorías de producto se tomarán del catálogo, que funcionará como fuente de referencia. Los canales ausentes de marketing se recuperarán desde `id_campaña`.

In [26]:
# conservar copias de los datos originales
orders_original = orders.copy()
catalog_original = catalog.copy()
marketing_original = marketing.copy()

# eliminar duplicados exactos de orders
orders = orders.drop_duplicates().copy()

# eliminar pedidos sin información indispensable
columnas_indispensables = [
    'nombre_producto',
    'cantidad',
    'precio_unitario',
    'monto_descuento'
]

orders = orders.dropna(
    subset=columnas_indispensables
).copy()

# estandarizar los países
orders['pais'] = (
    orders['pais']
    .str.capitalize()
    .fillna('Desconocido')
)

# tratar dispositivos ausentes
orders['dispositivo'] = (
    orders['dispositivo']
    .fillna('desconocido')
)

# utilizar el catálogo como referencia para la categoría
mapa_categorias = catalog.set_index(
    'nombre_producto'
)['categoria_producto']

orders['categoria_producto'] = (
    orders['nombre_producto']
    .map(mapa_categorias)
)

# identificar cantidades negativas y cantidades con error de escala
filtro_cantidades_negativas = orders['cantidad'] < 0
filtro_error_escala = orders['cantidad'] >= 10000

print(
    'Cantidades negativas corregidas:',
    filtro_cantidades_negativas.sum()
)

print(
    'Cantidades con error de escala corregidas:',
    filtro_error_escala.sum()
)

# corregir cantidades negativas
orders.loc[
    filtro_cantidades_negativas,
    'cantidad'
] = (
    orders.loc[
        filtro_cantidades_negativas,
        'cantidad'
    ].abs()
)

# corregir cantidades multiplicadas por 10.000
orders.loc[
    filtro_error_escala,
    'cantidad'
] = (
    orders.loc[
        filtro_error_escala,
        'cantidad'
    ] / 10000
)

# recalcular el total de los pedidos corregidos
filtro_cantidades_corregidas = (
    filtro_cantidades_negativas |
    filtro_error_escala
)

orders.loc[
    filtro_cantidades_corregidas,
    'monto_total'
] = (
    orders.loc[
        filtro_cantidades_corregidas,
        'cantidad'
    ] *
    orders.loc[
        filtro_cantidades_corregidas,
        'precio_unitario'
    ] -
    orders.loc[
        filtro_cantidades_corregidas,
        'monto_descuento'
    ]
).round(2)

# convertir cantidad a número entero
orders['cantidad'] = orders['cantidad'].astype(int)

# recuperar los canales ausentes de marketing
canal_inferido = (
    marketing['id_campaña']
    .str.rsplit('_', n=1)
    .str[0]
)

marketing['canal'] = (
    marketing['canal']
    .fillna(canal_inferido)
)

print('\nDimensiones después de la limpieza inicial:')
print('Orders:', orders.shape)
print('Catalog:', catalog.shape)
print('Marketing:', marketing.shape)

Cantidades negativas corregidas: 4
Cantidades con error de escala corregidas: 10

Dimensiones después de la limpieza inicial:
Orders: (24920, 12)
Catalog: (7, 4)
Marketing: (1620, 5)


### Resultado de la limpieza

Se eliminaron 100 copias duplicadas y 80 pedidos que no contenían información suficiente para calcular correctamente la rentabilidad. El dataset `orders` pasó de 25.100 a 24.920 filas.

Se corrigieron cuatro cantidades negativas y diez cantidades afectadas por un error de escala. Los países fueron estandarizados, las categorías de producto se asignaron utilizando el catálogo como fuente de referencia y los canales ausentes de marketing se recuperaron desde `id_campaña`.

Los datasets `catalog` y `marketing` conservaron todas sus filas.

In [27]:
# validación final de los datasets limpios
monto_calculado_final = (
    orders['cantidad'] *
    orders['precio_unitario'] -
    orders['monto_descuento']
).round(2)

diferencia_monto_final = (
    orders['monto_total'] -
    monto_calculado_final
).abs().round(2)

categoria_catalogo = (
    orders['nombre_producto']
    .map(mapa_categorias)
)

validaciones = pd.Series({
    'Orders sin duplicados exactos':
        orders.duplicated().sum() == 0,

    'ID de pedido único':
        orders['id_pedido'].is_unique,

    'Sin nulos en columnas indispensables':
        orders[columnas_indispensables]
        .isna()
        .sum()
        .sum() == 0,

    'Cantidades únicamente 1 o 2':
        orders['cantidad']
        .isin([1, 2])
        .all(),

    'Sin precios iguales o menores que cero':
        (orders['precio_unitario'] > 0).all(),

    'Sin montos totales iguales o menores que cero':
        (orders['monto_total'] > 0).all(),

    'Montos consistentes con tolerancia de 0.01':
        (diferencia_monto_final <= 0.01).all(),

    'Todos los productos existen en catalog':
        orders['nombre_producto']
        .isin(catalog['nombre_producto'])
        .all(),

    'Categorías coinciden con catalog':
        orders['categoria_producto']
        .eq(categoria_catalogo)
        .all(),

    'Catalog sin duplicados':
        catalog.duplicated().sum() == 0,

    'Marketing sin duplicados':
        marketing.duplicated().sum() == 0,

    'Marketing sin canales ausentes':
        marketing['canal'].isna().sum() == 0,

    'Costos de catalog mayores que cero':
        (catalog['costo_unitario'] > 0).all(),

    'Gastos de marketing mayores que cero':
        (marketing['gasto'] > 0).all()
}).to_frame(name='resultado')

print('RESULTADO DE LAS VALIDACIONES')
display(validaciones)

print('\nVALORES AUSENTES DESPUÉS DE LA LIMPIEZA')

print('\nOrders:')
display(orders.isna().sum().to_frame(name='cantidad_nulos'))

print('Catalog:')
display(catalog.isna().sum().to_frame(name='cantidad_nulos'))

print('Marketing:')
display(marketing.isna().sum().to_frame(name='cantidad_nulos'))

print('CATEGORÍAS CONSERVADAS COMO DESCONOCIDAS')

print('\nPaís:')
display(
    orders['pais']
    .value_counts()
    .to_frame(name='frecuencia')
)

print('Dispositivo:')
display(
    orders['dispositivo']
    .value_counts()
    .to_frame(name='frecuencia')
)

RESULTADO DE LAS VALIDACIONES


,resultado
Orders sin duplicados exactos,True
ID de pedido único,True
Sin nulos en columnas indispensables,True
Cantidades únicamente 1 o 2,True
Sin precios iguales o menores que cero,True
Sin montos totales iguales o menores que cero,True
Montos consistentes con tolerancia de 0.01,True
Todos los productos existen en catalog,True
Categorías coinciden con catalog,True
Catalog sin duplicados,True



VALORES AUSENTES DESPUÉS DE LA LIMPIEZA

Orders:


,cantidad_nulos
id_pedido,0
id_usuario,0
fecha_hora_pedido,0
pais,0
dispositivo,0
fuente_referencia,0
nombre_producto,0
categoria_producto,0
cantidad,0
precio_unitario,0


Catalog:


...

,cantidad_nulos
fecha,0
pais,0
id_campaña,0
canal,0
gasto,0


CATEGORÍAS CONSERVADAS COMO DESCONOCIDAS

País:


,frecuencia
Mexico,8313
Colombia,8282
Argentina,8025
Desconocido,300


Dispositivo:


,frecuencia
desktop,12670
mobile,12230
desconocido,20


### Conclusión de la revisión y limpieza de datos

Se revisaron y limpiaron los tres datasets utilizados en el proyecto.

En `orders` se eliminaron 100 registros duplicados exactos y 80 filas que no contenían información indispensable para el análisis. También se corrigieron cuatro cantidades negativas y diez cantidades con errores de escala. Después de estas correcciones, los montos totales fueron recalculados y validados.

Los nombres de países fueron estandarizados, las categorías de productos se recuperaron utilizando el catálogo y los valores de país y dispositivo que no podían determinarse se conservaron como `Desconocido`.

En `marketing`, los 101 canales ausentes fueron recuperados a partir del identificador de campaña. El dataset `catalog` no presentó duplicados ni valores ausentes.

Después de la limpieza:

- `orders` contiene 24.920 registros.
- `catalog` contiene 7 registros.
- `marketing` contiene 1.620 registros.
- No quedan valores ausentes.
- No quedan duplicados.
- Las cantidades, precios, costos, gastos y montos cumplen las reglas de validación.
- Los productos y categorías son consistentes con el catálogo.

Los datasets están preparados para el análisis de rentabilidad y para las siguientes etapas del proyecto.

In [28]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

In [29]:
# comprobar los archivos exportados
orders_exportado = pd.read_csv('orders_clean.csv')
catalog_exportado = pd.read_csv('catalog_clean.csv')
marketing_exportado = pd.read_csv('marketing_clean.csv')

print('ARCHIVOS EXPORTADOS CORRECTAMENTE')
print('orders_clean.csv:', orders_exportado.shape)
print('catalog_clean.csv:', catalog_exportado.shape)
print('marketing_clean.csv:', marketing_exportado.shape)

ARCHIVOS EXPORTADOS CORRECTAMENTE
orders_clean.csv: (24920, 12)
catalog_clean.csv: (7, 4)
marketing_clean.csv: (1620, 5)


## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [30]:
# unir los pedidos con el costo unitario del catálogo
orders_rentabilidad = orders.merge(
    catalog[
        [
            'nombre_producto',
            'costo_unitario'
        ]
    ],
    on='nombre_producto',
    how='left',
    validate='many_to_one'
)

# comprobar el resultado de la unión
print('Dimensiones de orders:', orders.shape)
print(
    'Dimensiones de orders_rentabilidad:',
    orders_rentabilidad.shape
)

print(
    'Costos unitarios ausentes:',
    orders_rentabilidad['costo_unitario'].isna().sum()
)

display(
    orders_rentabilidad[
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total',
            'costo_unitario'
        ]
    ].head()
)


Dimensiones de orders: (24920, 12)
Dimensiones de orders_rentabilidad: (24920, 13)
Costos unitarios ausentes: 0


,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario
0,order_0,Jacket-Winter-M,2,332.69,0.0,665.37,189.31
1,order_1,Tablet-Standard-64GB,1,176.86,5.0,171.86,25.21
2,order_2,Blender-XL-Red,2,102.99,10.0,195.99,176.64
3,order_3,Tablet-Standard-64GB,1,257.87,15.0,242.87,25.21
4,order_4,Blender-XL-Red,1,336.28,0.0,336.28,176.64


In [31]:
# calcular el costo de los productos de cada pedido
orders_rentabilidad['costo_producto'] = (
    orders_rentabilidad['cantidad'] *
    orders_rentabilidad['costo_unitario']
).round(2)

# calcular el beneficio bruto de cada pedido
orders_rentabilidad['beneficio_bruto'] = (
    orders_rentabilidad['monto_total'] -
    orders_rentabilidad['costo_producto']
).round(2)

# Parte 1: KPIs de rentabilidad
ingreso_total = (
    orders_rentabilidad['monto_total'].sum()
)

costo_total = (
    orders_rentabilidad['costo_producto'].sum()
)

inversion_marketing = (
    marketing['gasto'].sum()
)

costo_total_negocio = (
    costo_total +
    inversion_marketing
)

profit_total = (
    ingreso_total -
    costo_total -
    inversion_marketing
)

margen_profit = (
    profit_total /
    ingreso_total *
    100
)

# mostrar respuestas de la Parte 1
print('PARTE 1: RENTABILIDAD DEL NEGOCIO')
print(f'1. Ingreso total (revenue): {ingreso_total:,.2f}')
print(f'2. Costo total de productos: {costo_total:,.2f}')
print(f'3. Inversión en marketing: {inversion_marketing:,.2f}')
print(
    f'   Costo total del negocio '
    f'(productos + marketing): {costo_total_negocio:,.2f}'
)
print(f'4. Profit total: {profit_total:,.2f}')
print(f'   Margen de profit: {margen_profit:.2f}%')

if profit_total > 0:
    print('   Conclusión: el negocio es rentable.')
elif profit_total < 0:
    print('   Conclusión: el negocio no es rentable.')
else:
    print('   Conclusión: el negocio está en punto de equilibrio.')

PARTE 1: RENTABILIDAD DEL NEGOCIO
1. Ingreso total (revenue): 9,615,400.71
2. Costo total de productos: 3,832,849.13
3. Inversión en marketing: 2,871,843.53
   Costo total del negocio (productos + marketing): 6,704,692.66
4. Profit total: 2,910,708.05
   Margen de profit: 30.27%
   Conclusión: el negocio es rentable.


### Análisis de la Parte 1: rentabilidad del negocio

Durante el periodo analizado, el negocio generó ingresos totales de **9.615.400,71**. El costo de los productos vendidos fue de **3.832.849,13** y la inversión total en marketing alcanzó **2.871.843,53**.

Al sumar el costo de los productos y la inversión en marketing, el costo total considerado para el negocio fue de **6.704.692,66**. Después de descontar estos costos de los ingresos, se obtuvo un profit de **2.910.708,05**, equivalente a un margen de profit del **30,27 %**.

Por lo tanto, el negocio fue rentable durante el periodo analizado, ya que generó un resultado positivo después de cubrir los costos de los productos y la inversión en marketing.

Este resultado considera únicamente los costos disponibles en los datasets. No incluye otros posibles gastos operativos, como salarios, logística, impuestos o infraestructura.

In [32]:
# Parte 2: comportamiento de ventas

# agrupar la información a nivel de pedido
ventas_por_pedido = (
    orders_rentabilidad
    .groupby('id_pedido', as_index=False)
    .agg(
        monto_pedido=('monto_total', 'sum'),
        productos_pedido=('cantidad', 'sum')
    )
)

# calcular ticket y cantidad promedio por pedido
ticket_promedio = (
    ventas_por_pedido['monto_pedido'].mean()
)

cantidad_promedio = (
    ventas_por_pedido['productos_pedido'].mean()
)

# calcular unidades vendidas por producto
ventas_por_producto = (
    orders_rentabilidad
    .groupby('nombre_producto', as_index=False)
    .agg(unidades_vendidas=('cantidad', 'sum'))
    .sort_values(
        'unidades_vendidas',
        ascending=False
    )
    .reset_index(drop=True)
)

producto_mas_vendido = (
    ventas_por_producto.iloc[0]['nombre_producto']
)

unidades_producto_lider = (
    ventas_por_producto.iloc[0]['unidades_vendidas']
)

# calcular gasto de marketing por canal
marketing_por_canal = (
    marketing
    .groupby('canal', as_index=False)
    .agg(gasto_total=('gasto', 'sum'))
    .sort_values(
        'gasto_total',
        ascending=False
    )
    .reset_index(drop=True)
)

# mostrar respuestas
print('PARTE 2: COMPORTAMIENTO DE VENTAS')
print(f'1. Ticket promedio por orden: {ticket_promedio:,.2f}')
print(
    '2. Cantidad promedio de productos por orden: '
    f'{cantidad_promedio:.2f}'
)
print(
    f'3. Producto más vendido: {producto_mas_vendido} '
    f'({unidades_producto_lider:,.0f} unidades)'
)

print('\nVENTAS POR PRODUCTO')
display(ventas_por_producto)

print('4. GASTO DE MARKETING POR CANAL')
display(marketing_por_canal)

PARTE 2: COMPORTAMIENTO DE VENTAS
1. Ticket promedio por orden: 385.85
2. Cantidad promedio de productos por orden: 1.50
3. Producto más vendido: Vacuum-Pro-Black (6,284 unidades)

VENTAS POR PRODUCTO


,nombre_producto,unidades_vendidas
0,Vacuum-Pro-Black,6284
1,Blender-XL-Red,6279
2,Jacket-Winter-M,6256
3,Sneakers-Urban-42,6172
4,Laptop-Gaming-16GB,4212
5,Tablet-Standard-64GB,4153
6,Phone-Pro-128GB,4145


4. GASTO DE MARKETING POR CANAL


,canal,gasto_total
0,social,976818.37
1,organic,972650.96
2,paid_search,922374.20


In [33]:
# validación final de los KPIs del Paso 2
validaciones_paso_2 = pd.Series({
    'El ingreso coincide con la suma de pedidos':
        round(ventas_por_pedido['monto_pedido'].sum(), 2)
        == round(ingreso_total, 2),

    'Las unidades por producto coinciden con orders':
        ventas_por_producto['unidades_vendidas'].sum()
        == orders_rentabilidad['cantidad'].sum(),

    'El marketing por canal coincide con el total':
        round(marketing_por_canal['gasto_total'].sum(), 2)
        == round(inversion_marketing, 2),

    'El costo total del negocio está bien calculado':
        round(costo_total + inversion_marketing, 2)
        == round(costo_total_negocio, 2),

    'El profit está bien calculado':
        round(
            ingreso_total -
            costo_total -
            inversion_marketing,
            2
        )
        == round(profit_total, 2),

    'El ticket promedio es positivo':
        ticket_promedio > 0,

    'La cantidad promedio es positiva':
        cantidad_promedio > 0
}).to_frame(name='resultado')

print('VALIDACIÓN FINAL DEL PASO 2')
display(validaciones_paso_2)

VALIDACIÓN FINAL DEL PASO 2


,resultado
El ingreso coincide con la suma de pedidos,True
Las unidades por producto coinciden con orders,True
El marketing por canal coincide con el total,True
El costo total del negocio está bien calculado,True
El profit está bien calculado,True
El ticket promedio es positivo,True
La cantidad promedio es positiva,True


### Conclusión del Paso 2: rentabilidad y comportamiento de ventas

#### Parte 1: rentabilidad del negocio

Durante el periodo analizado, el negocio obtuvo los siguientes resultados:

- **Ingreso total (revenue):** 9.615.400,71.
- **Costo total de los productos vendidos:** 3.832.849,13.
- **Inversión total en marketing:** 2.871.843,53.
- **Costo total del negocio, incluyendo productos y marketing:** 6.704.692,66.
- **Profit:** 2.910.708,05.
- **Margen de profit:** 30,27 %.

El negocio fue rentable durante el periodo analizado, ya que los ingresos fueron superiores a la suma del costo de los productos y la inversión en marketing.

Esta conclusión considera únicamente los costos disponibles en los datasets. No incluye otros posibles gastos operativos, como salarios, logística, impuestos o infraestructura.

#### Parte 2: comportamiento de ventas

- **Ticket promedio por orden:** 385,85.
- **Cantidad promedio de productos por orden:** 1,50 unidades.
- **Producto más vendido:** `Vacuum-Pro-Black`, con 6.284 unidades.
- **Gasto de marketing por canal:**
  - `social`: 976.818,37.
  - `organic`: 972.650,96.
  - `paid_search`: 922.374,20.

`Vacuum-Pro-Black` fue el producto con mayor cantidad de unidades vendidas, aunque superó a `Blender-XL-Red` por solo cinco unidades. Esto indica que no existe una diferencia amplia entre los dos productos líderes.

El canal `social` recibió la mayor inversión de marketing, seguido muy de cerca por `organic`. Sin embargo, el gasto por sí solo no permite determinar qué canal fue más eficiente o rentable; para ello sería necesario relacionar la inversión con los ingresos o conversiones atribuidos a cada canal.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
# Configurar una conexión propia; no publicar credenciales.
import os
import pandas as pd
from sqlalchemy import create_engine

database_url = os.environ.get('RAPPIPLUS_DATABASE_URL')
if not database_url:
    raise RuntimeError(
        'Configura RAPPIPLUS_DATABASE_URL para ejecutar las secciones SQL. '
        'Los resultados guardados se pueden consultar sin conexión.'
    )
engine = create_engine(database_url)


In [35]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [36]:
# identificar los eventos y sus usuarios únicos
query_etapas = '''
SELECT
    nombre_evento,
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''

etapas = pd.read_sql(query_etapas, con=engine)
etapas

,nombre_evento,total_eventos,usuarios_unicos
0,first_visit,29957,7796
1,add_to_cart,24157,7634
2,select_item,23887,7582
3,begin_checkout,17971,7208
4,add_payment_info,12018,6250
5,purchase,12010,6240


In [38]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
        ELSE 7
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [39]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
            ELSE 7
        END AS orden_etapa,
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
),

funnel_con_etapa_anterior AS (
    SELECT
        orden_etapa,
        nombre_evento,
        usuarios_unicos,
        LAG(nombre_evento) OVER (
            ORDER BY orden_etapa
        ) AS etapa_anterior,
        LAG(usuarios_unicos) OVER (
            ORDER BY orden_etapa
        ) AS usuarios_etapa_anterior
    FROM funnel
)

SELECT
    orden_etapa,
    nombre_evento,
    usuarios_unicos,
    etapa_anterior,
    usuarios_etapa_anterior,

    CASE
        WHEN usuarios_etapa_anterior IS NULL THEN NULL
        ELSE ROUND(
            100.0 * usuarios_unicos /
            usuarios_etapa_anterior,
            2
        )
    END AS tasa_conversion_pct,

    CASE
        WHEN usuarios_etapa_anterior IS NULL THEN NULL
        ELSE usuarios_etapa_anterior - usuarios_unicos
    END AS usuarios_perdidos,

    CASE
        WHEN usuarios_etapa_anterior IS NULL THEN NULL
        ELSE ROUND(
            100.0 *
            (usuarios_etapa_anterior - usuarios_unicos) /
            usuarios_etapa_anterior,
            2
        )
    END AS tasa_abandono_pct

FROM funnel_con_etapa_anterior
ORDER BY orden_etapa;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,orden_etapa,nombre_evento,usuarios_unicos,etapa_anterior,usuarios_etapa_anterior,tasa_conversion_pct,usuarios_perdidos,tasa_abandono_pct
0,1,first_visit,7796,None,NaN,NaN,NaN,NaN
1,2,select_item,7582,first_visit,7796.0,97.26,214.0,2.74
2,3,add_to_cart,7634,select_item,7582.0,100.69,-52.0,-0.69
3,4,begin_checkout,7208,add_to_cart,7634.0,94.42,426.0,5.58
4,5,add_payment_info,6250,begin_checkout,7208.0,86.71,958.0,13.29
5,6,purchase,6240,add_payment_info,6250.0,99.84,10.0,0.16


---

In [40]:
# resumen final del funnel

# recuperar usuarios de la primera y última etapa
usuarios_iniciales = totals.loc[
    totals['nombre_evento'] == 'first_visit',
    'usuarios_unicos'
].iloc[0]

usuarios_compradores = totals.loc[
    totals['nombre_evento'] == 'purchase',
    'usuarios_unicos'
].iloc[0]

# calcular la conversión final
conversion_final = (
    usuarios_compradores /
    usuarios_iniciales *
    100
)

# excluir la anomalía negativa para buscar pérdidas reales
mayor_abandono = (
    conversion[
        conversion['usuarios_perdidos'] >= 0
    ]
    .sort_values(
        'usuarios_perdidos',
        ascending=False
    )
    .iloc[0]
)

# mostrar respuestas finales
print('RESUMEN DEL FUNNEL')
print(f'Usuarios en first_visit: {usuarios_iniciales:,}')
print(f'Usuarios en purchase: {usuarios_compradores:,}')
print(f'Conversión final: {conversion_final:.2f}%')

print('\nMAYOR ABANDONO')
print(
    f"Transición: {mayor_abandono['etapa_anterior']} "
    f"→ {mayor_abandono['nombre_evento']}"
)
print(
    'Usuarios perdidos: '
    f"{mayor_abandono['usuarios_perdidos']:,.0f}"
)
print(
    'Tasa de abandono: '
    f"{mayor_abandono['tasa_abandono_pct']:.2f}%"
)

RESUMEN DEL FUNNEL
Usuarios en first_visit: 7,796
Usuarios en purchase: 6,240
Conversión final: 80.04%

MAYOR ABANDONO
Transición: begin_checkout → add_payment_info
Usuarios perdidos: 958
Tasa de abandono: 13.29%


### Conclusión del Paso 3: funnel de conversión

El funnel analizado contiene las siguientes etapas:

1. `first_visit`: 7.796 usuarios.
2. `select_item`: 7.582 usuarios.
3. `add_to_cart`: 7.634 usuarios.
4. `begin_checkout`: 7.208 usuarios.
5. `add_payment_info`: 6.250 usuarios.
6. `purchase`: 6.240 usuarios.

La conversión final desde `first_visit` hasta `purchase` fue del **80,04 %**, ya que 6.240 de los 7.796 usuarios contabilizados en la primera etapa llegaron al evento de compra.

El mayor punto de abandono se encontró en la transición de `begin_checkout` a `add_payment_info`. En esta etapa se perdieron **958 usuarios**, equivalentes a una tasa de abandono del **13,29 %**. Este resultado indica que la incorporación de los datos de pago representa la principal fricción del proceso de compra.

Después de introducir la información de pago, la conversión hacia `purchase` fue del **99,84 %**, por lo que casi todos los usuarios que superan la etapa de pago completan la compra.

Se detectó una anomalía entre `select_item` y `add_to_cart`: `add_to_cart` registra 52 usuarios más y una conversión aparente del 100,69 %. Esto indica que los conteos independientes por evento no forman un recorrido estrictamente decreciente. Puede deberse a recorridos alternativos, accesos directos al carrito o eventos no registrados.

Como recomendación, se debería revisar la experiencia de ingreso de datos de pago, incluyendo posibles errores técnicos, cantidad de campos, métodos de pago disponibles y claridad de las instrucciones.

Las tasas se calcularon utilizando usuarios únicos por evento. Por tanto, describen el alcance de cada etapa, pero no comprueban individualmente que cada usuario haya realizado todas las etapas anteriores en el orden esperado.

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [41]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [43]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT*
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [44]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        DATE_TRUNC(
            'month',
            CAST(fecha_registro AS DATE)
        )::DATE AS cohorte
    FROM users
),

retencion_por_cohorte AS (
    SELECT
        uc.cohorte,

        COUNT(
            DISTINCT uc.id_usuario
        ) AS clientes_iniciales,

        COUNT(
            DISTINCT CASE
                WHEN ua.dias_despues_registro = 7
                     AND ua.activo = 1
                THEN uc.id_usuario
            END
        ) AS retenido_w1,

        COUNT(
            DISTINCT CASE
                WHEN ua.dias_despues_registro = 14
                     AND ua.activo = 1
                THEN uc.id_usuario
            END
        ) AS retenido_w2,

        COUNT(
            DISTINCT CASE
                WHEN ua.dias_despues_registro = 21
                     AND ua.activo = 1
                THEN uc.id_usuario
            END
        ) AS retenido_w3

    FROM usuarios_cohorte AS uc

    LEFT JOIN user_activity AS ua
        ON uc.id_usuario = ua.id_usuario

    GROUP BY uc.cohorte
)

SELECT
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,

    ROUND(
        100.0 * retenido_w1 /
        NULLIF(clientes_iniciales, 0),
        2
    ) AS semana_1,

    ROUND(
        100.0 * retenido_w2 /
        NULLIF(clientes_iniciales, 0),
        2
    ) AS semana_2,

    ROUND(
        100.0 * retenido_w3 /
        NULLIF(clientes_iniciales, 0),
        2
    ) AS semana_3

FROM retencion_por_cohorte
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01,1687,695,676,706,41.20,40.07,41.85


In [45]:
# validar y resumir la retención por cohortes

usuarios_totales = users['id_usuario'].nunique()

total_retenido_w1 = (
    cohorte_final['retenido_w1'].sum()
)

total_retenido_w2 = (
    cohorte_final['retenido_w2'].sum()
)

total_retenido_w3 = (
    cohorte_final['retenido_w3'].sum()
)

retencion_general_w1 = (
    total_retenido_w1 /
    usuarios_totales *
    100
)

retencion_general_w2 = (
    total_retenido_w2 /
    usuarios_totales *
    100
)

retencion_general_w3 = (
    total_retenido_w3 /
    usuarios_totales *
    100
)

validaciones_retencion = pd.Series({
    'Las cohortes incluyen a todos los usuarios':
        cohorte_final['clientes_iniciales'].sum()
        == usuarios_totales,

    'Retenidos W1 no superan clientes iniciales':
        (
            cohorte_final['retenido_w1']
            <= cohorte_final['clientes_iniciales']
        ).all(),

    'Retenidos W2 no superan clientes iniciales':
        (
            cohorte_final['retenido_w2']
            <= cohorte_final['clientes_iniciales']
        ).all(),

    'Retenidos W3 no superan clientes iniciales':
        (
            cohorte_final['retenido_w3']
            <= cohorte_final['clientes_iniciales']
        ).all(),

    'Porcentajes W1 están entre 0 y 100':
        cohorte_final['semana_1'].between(0, 100).all(),

    'Porcentajes W2 están entre 0 y 100':
        cohorte_final['semana_2'].between(0, 100).all(),

    'Porcentajes W3 están entre 0 y 100':
        cohorte_final['semana_3'].between(0, 100).all()
}).to_frame(name='resultado')

print('VALIDACIONES DE RETENCIÓN')
display(validaciones_retencion)

print('RESUMEN GENERAL')
print(f'Usuarios analizados: {usuarios_totales:,}')
print(f'Retención general semana 1: {retencion_general_w1:.2f}%')
print(f'Retención general semana 2: {retencion_general_w2:.2f}%')
print(f'Retención general semana 3: {retencion_general_w3:.2f}%')

VALIDACIONES DE RETENCIÓN


,resultado
Las cohortes incluyen a todos los usuarios,True
Retenidos W1 no superan clientes iniciales,True
Retenidos W2 no superan clientes iniciales,True
Retenidos W3 no superan clientes iniciales,True
Porcentajes W1 están entre 0 y 100,True
Porcentajes W2 están entre 0 y 100,True
Porcentajes W3 están entre 0 y 100,True


RESUMEN GENERAL
Usuarios analizados: 8,000
Retención general semana 1: 42.00%
Retención general semana 2: 41.94%
Retención general semana 3: 41.88%


### Conclusión del Paso 4: retención por cohortes

Se analizaron 8.000 usuarios, agrupados en cinco cohortes según su mes de registro, desde enero hasta mayo de 2025.

La retención general fue:

- **Semana 1:** 42,00 %.
- **Semana 2:** 41,94 %.
- **Semana 3:** 41,88 %.

La retención se mantuvo prácticamente estable durante las tres semanas. Entre la semana 1 y la semana 3 se produjo una disminución de solo **0,12 puntos porcentuales**, por lo que no se observa un abandono progresivo pronunciado durante este periodo.

La cohorte de febrero presentó la mayor retención en la semana 3, con un **43,98 %**. La cohorte de enero presentó la menor retención en esa semana, con un **40,32 %**.

La evolución no es necesariamente descendente en cada cohorte. Por ejemplo, algunos meses presentan más usuarios activos en la semana 3 que en la semana 2. Esto es posible porque la variable `activo` mide la actividad de cada semana de manera independiente: un usuario puede estar inactivo durante una semana y regresar posteriormente.

Aunque la estabilidad es positiva, alrededor del 58 % de los usuarios no aparece activo en cada una de las semanas analizadas. Se recomienda investigar los motivos de inactividad y aplicar estrategias de reactivación, como comunicaciones personalizadas, recordatorios, promociones o mejoras en la experiencia inicial.

También sería conveniente segmentar posteriormente la retención por país, dispositivo y tipo de plan para identificar grupos con mejores o peores resultados.

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

Las hipótesis y el método estadístico se detallan junto al cálculo del test Z, a continuación.


In [46]:
# cargar el dataset del experimento A/B
experiment = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/'
    'datasets/experiment_checkout_ui.csv'
)

# convertir la fecha al formato correcto
experiment['timestamp'] = pd.to_datetime(
    experiment['timestamp'],
    errors='coerce'
)

# revisión inicial
print('DIMENSIONES DEL DATASET')
print(experiment.shape)

print('\nPRIMERAS FILAS')
display(experiment.head())

print('VALORES AUSENTES')
display(
    experiment
    .isna()
    .sum()
    .to_frame(name='cantidad_nulos')
)

print('DISTRIBUCIÓN DE LOS GRUPOS')
display(
    experiment['variante']
    .value_counts()
    .to_frame(name='usuarios')
)

print('VALORES DE LA VARIABLE CONVIRTIO')
display(
    experiment['convirtio']
    .value_counts()
    .sort_index()
    .to_frame(name='usuarios')
)

# comprobaciones del diseño experimental
duplicados = experiment.duplicated().sum()
usuarios_unicos = experiment['id_usuario'].nunique()

usuarios_en_ambos_grupos = (
    experiment
    .groupby('id_usuario')['variante']
    .nunique()
    .gt(1)
    .sum()
)

print('CONTROL DEL EXPERIMENTO')
print(f'Filas duplicadas: {duplicados}')
print(f'Filas totales: {experiment.shape[0]:,}')
print(f'Usuarios únicos: {usuarios_unicos:,}')
print(
    'Usuarios presentes en ambos grupos: '
    f'{usuarios_en_ambos_grupos}'
)
print(
    'Periodo del experimento:',
    experiment['timestamp'].min(),
    '-',
    experiment['timestamp'].max()
)



DIMENSIONES DEL DATASET
(10000, 7)

PRIMERAS FILAS


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


VALORES AUSENTES


,cantidad_nulos
id_usuario,0
variante,0
convirtio,0
dispositivo,0
pais,0
duracion_sesion,0
timestamp,0


DISTRIBUCIÓN DE LOS GRUPOS


,usuarios
tratamiento,5035
control,4965


VALORES DE LA VARIABLE CONVIRTIO


,usuarios
0,8401
1,1599


CONTROL DEL EXPERIMENTO
Filas duplicadas: 0
Filas totales: 10,000
Usuarios únicos: 10,000
Usuarios presentes en ambos grupos: 0
Periodo del experimento: 2025-01-01 00:00:00 - 2025-06-30 00:00:00


### Hipótesis estadística

- **H₀ (hipótesis nula):** La tasa de conversión del grupo de tratamiento es igual a la tasa de conversión del grupo de control. La modificación de la interfaz del checkout no produjo un cambio estadísticamente significativo.

- **H₁ (hipótesis alternativa):** La tasa de conversión del grupo de tratamiento es diferente de la tasa de conversión del grupo de control. La modificación de la interfaz del checkout produjo un cambio estadísticamente significativo.

**Test estadístico:** prueba Z para comparar dos proporciones independientes.

**Nivel de significancia:** α = 0,05.

In [47]:
# calcular la conversión observada de cada grupo
resumen_experimento = (
    experiment
    .groupby('variante')
    .agg(
        usuarios=('id_usuario', 'nunique'),
        conversiones=('convirtio', 'sum'),
        tasa_conversion=('convirtio', 'mean')
    )
    .reset_index()
)

resumen_experimento['tasa_conversion_pct'] = (
    resumen_experimento['tasa_conversion'] *
    100
)

# recuperar las tasas para comparar los grupos
tasa_control = resumen_experimento.loc[
    resumen_experimento['variante'] == 'control',
    'tasa_conversion'
].iloc[0]

tasa_tratamiento = resumen_experimento.loc[
    resumen_experimento['variante'] == 'tratamiento',
    'tasa_conversion'
].iloc[0]

diferencia_pp = (
    tasa_tratamiento -
    tasa_control
) * 100

uplift_relativo = (
    (tasa_tratamiento - tasa_control) /
    tasa_control *
    100
)

print('RESULTADOS OBSERVADOS DEL EXPERIMENTO')
display(
    resumen_experimento[
        [
            'variante',
            'usuarios',
            'conversiones',
            'tasa_conversion_pct'
        ]
    ]
)

print(
    'Diferencia tratamiento - control: '
    f'{diferencia_pp:.2f} puntos porcentuales'
)

print(
    'Uplift relativo del tratamiento: '
    f'{uplift_relativo:.2f}%'
)

RESULTADOS OBSERVADOS DEL EXPERIMENTO


,variante,usuarios,conversiones,tasa_conversion_pct
0,control,4965,779,15.689829
1,tratamiento,5035,820,16.285998


Diferencia tratamiento - control: 0.60 puntos porcentuales
Uplift relativo del tratamiento: 3.80%


In [48]:
# Test Z para comparar dos proporciones
import numpy as np
from scipy.stats import norm

alpha = 0.05

# Datos de cada variante
n_control = 4965
conversiones_control = 779

n_tratamiento = 5035
conversiones_tratamiento = 820

# Tasas de conversión
p_control = conversiones_control / n_control
p_tratamiento = conversiones_tratamiento / n_tratamiento

# Proporción combinada bajo la hipótesis nula
p_combinada = (
    conversiones_control + conversiones_tratamiento
) / (
    n_control + n_tratamiento
)

# Error estándar combinado
error_estandar = np.sqrt(
    p_combinada
    * (1 - p_combinada)
    * (1 / n_control + 1 / n_tratamiento)
)

# Estadístico Z y p-value bilateral
z_stat = (p_tratamiento - p_control) / error_estandar
p_value = 2 * (1 - norm.cdf(abs(z_stat)))

# Intervalo de confianza del 95% para la diferencia
diferencia = p_tratamiento - p_control

error_diferencia = np.sqrt(
    p_tratamiento * (1 - p_tratamiento) / n_tratamiento
    + p_control * (1 - p_control) / n_control
)

z_critico = norm.ppf(1 - alpha / 2)

limite_inferior = diferencia - z_critico * error_diferencia
limite_superior = diferencia + z_critico * error_diferencia

print('RESULTADO DEL TEST A/B')
print(f'Estadístico Z: {z_stat:.4f}')
print(f'P-value: {p_value:.4f}')
print(f'Nivel de significancia: {alpha}')
print(
    'Intervalo de confianza del 95%: '
    f'[{limite_inferior * 100:.2f}, '
    f'{limite_superior * 100:.2f}] puntos porcentuales'
)

if p_value < alpha:
    print('Decisión: se rechaza H0.')
    print('La diferencia es estadísticamente significativa.')
else:
    print('Decisión: no se rechaza H0.')
    print('La diferencia no es estadísticamente significativa.')

RESULTADO DEL TEST A/B
Estadístico Z: 0.8133
P-value: 0.4161
Nivel de significancia: 0.05
Intervalo de confianza del 95%: [-0.84, 2.03] puntos porcentuales
Decisión: no se rechaza H0.
La diferencia no es estadísticamente significativa.


### Conclusión del Paso 5: experimento A/B

El grupo de control obtuvo una tasa de conversión del **15.69%**, mientras que el grupo de tratamiento alcanzó un **16.29%**.

La nueva interfaz presentó un aumento observado de **0.60 puntos porcentuales**, equivalente a un uplift relativo del **3.80%**. Sin embargo, el test Z para dos proporciones obtuvo:

- **Estadístico Z:** 0.8133
- **P-value:** 0.4161
- **Nivel de significancia:** 0.05
- **Intervalo de confianza del 95%:** entre −0.84 y 2.03 puntos porcentuales

Como el p-value es mayor que 0.05, **no se rechaza la hipótesis nula**. La diferencia observada no es estadísticamente significativa y puede explicarse por la variabilidad aleatoria de la muestra.

Por lo tanto, no existe evidencia suficiente para concluir que la modificación de la interfaz aumente la conversión. No se recomienda implementarla de manera general basándose únicamente en este experimento. Sería conveniente mantener la versión actual o realizar una prueba adicional con mayor tamaño de muestra antes de tomar una decisión definitiva.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

# https://drive.google.com/drive/folders/12iljaZl3qqyvo2uHi_BzJCfsjednBzVT?usp=sharing
